In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("C:/Users/ADMIN/Downloads/timeserries_models_analysis/assets/metrics_summary.csv")

# Sort for cleaner comparison
results_df = df.sort_values(
    by=["dataset", "rmse", "mae"],
    ascending=[True, True, True]
).reset_index(drop=True)

results_df

,dataset,model,rmse,mae,best_val_loss,epochs_ran
0,apple,transformer,0.569930,0.386512,0.271322,11
1,apple,lstm,0.575272,0.399727,0.278414,16
2,apple,rnn,0.609283,0.420242,0.280925,21
3,apple,mlp,1.174122,0.879710,0.304408,12
4,weather,rnn,0.036063,0.026517,0.000480,36
5,weather,lstm,0.048182,0.038324,0.000564,23
6,weather,mlp,0.072057,0.056606,0.004906,16
7,weather,transformer,0.084647,0.068755,0.000836,50


In [2]:
results_df = results_df.sort_values(
    by=["dataset", "rmse", "mae"],
    ascending=[True, True, True]
).reset_index(drop=True)

# Add ranking columns within each dataset
results_df["rmse_rank"] = results_df.groupby("dataset")["rmse"].rank(method="dense")
results_df["mae_rank"] = results_df.groupby("dataset")["mae"].rank(method="dense")

cols = ["dataset", "model", "rmse", "mae", "rmse_rank", "mae_rank"]
if "epochs_ran" in results_df.columns:
    cols.append("epochs_ran")

results_df = results_df[cols]

results_df

,dataset,model,rmse,mae,rmse_rank,mae_rank,epochs_ran
0,apple,transformer,0.569930,0.386512,1.0,1.0,11
1,apple,lstm,0.575272,0.399727,2.0,2.0,16
2,apple,rnn,0.609283,0.420242,3.0,3.0,21
3,apple,mlp,1.174122,0.879710,4.0,4.0,12
4,weather,rnn,0.036063,0.026517,1.0,1.0,36
5,weather,lstm,0.048182,0.038324,2.0,2.0,23
6,weather,mlp,0.072057,0.056606,3.0,3.0,16
7,weather,transformer,0.084647,0.068755,4.0,4.0,50


### Overall Results Ranking

The sorted results table provides a clear comparison of all models across both datasets using RMSE and MAE. A lower RMSE or MAE indicates better predictive performance, so sorting by these metrics helps identify the strongest model for each dataset.

For the Apple dataset, the Transformer ranks first with the lowest RMSE (0.5699) and MAE (0.3865), followed closely by the LSTM (RMSE = 0.5753, MAE = 0.3997). The RNN is slightly worse but still performs reasonably well, while the MLP ranks last by a large margin. This suggests that sequence-aware models are more effective for the Apple task, where temporal order matters and the target is based on log returns.

For the Weather dataset, the RNN performs best with the lowest RMSE (0.0361) and MAE (0.0265), followed by the LSTM. The MLP ranks third, while the Transformer ranks last. This indicates that although all models perform fairly well on the weather series, simpler recurrent architectures are better suited to capturing the structure of this dataset than the Transformer.

Overall, the ranking table shows that model performance depends strongly on the nature of the dataset. Sequence models dominate both datasets, but the exact best architecture differs depending on the characteristics of the time series.

In [3]:
apple_results = results_df[results_df["dataset"].str.lower() == "apple"].copy()
weather_results = results_df[results_df["dataset"].str.lower() == "weather"].copy()

print("Apple Results")
display(apple_results)

print("Weather Results")
display(weather_results)

Apple Results


,dataset,model,rmse,mae,rmse_rank,mae_rank,epochs_ran
0,apple,transformer,0.569930,0.386512,1.0,1.0,11
1,apple,lstm,0.575272,0.399727,2.0,2.0,16
2,apple,rnn,0.609283,0.420242,3.0,3.0,21
3,apple,mlp,1.174122,0.879710,4.0,4.0,12


Weather Results


,dataset,model,rmse,mae,rmse_rank,mae_rank,epochs_ran
4,weather,rnn,0.036063,0.026517,1.0,1.0,36
5,weather,lstm,0.048182,0.038324,2.0,2.0,23
6,weather,mlp,0.072057,0.056606,3.0,3.0,16
7,weather,transformer,0.084647,0.068755,4.0,4.0,50


### Apple Results

The Apple dataset results show that the Transformer achieved the best overall performance, with the lowest RMSE (0.5699) and MAE (0.3865). The LSTM ranked second and performed very similarly, with only a small difference in error compared to the Transformer. The RNN also performed reasonably well, but its error values were slightly higher, indicating that it captured the general temporal structure but was less accurate than the more advanced sequential models.

The MLP performed worst on the Apple dataset, with a noticeably higher RMSE (1.1741) and MAE (0.8797). This suggests that flattening the input window into a fixed vector loses important sequential information. Since the Apple target was transformed into log returns, the task became more dependent on modelling short-term temporal dependencies rather than simply following a long-term trend. This benefits models such as the RNN, LSTM, and Transformer, which are designed to handle ordered sequential inputs.

Another important observation is that the difference between the top three models is not extremely large, especially between the Transformer and LSTM. This suggests that both attention-based and recurrent approaches are effective for this task. However, the Transformer’s first-place ranking indicates that it was slightly better at capturing the short-term return dynamics in this particular setup.

In summary, the Apple results show that once the stock series is transformed into a more stationary target, sequence-based models become clearly more suitable than the MLP. The Transformer is the strongest model here, with the LSTM close behind.

### Weather Results

The Weather dataset shows a different ranking pattern from Apple. Here, the RNN achieved the best performance, with the lowest RMSE (0.0361) and MAE (0.0265). The LSTM ranked second and also performed strongly, with only slightly higher error values. These results indicate that recurrent models are very effective for this dataset, likely because the weather series contains stable temporal structure and repeating local patterns that are well captured by sequential memory-based architectures.

The MLP ranked third, performing worse than both recurrent models but still better than the Transformer. This suggests that while some predictive signal can be extracted without explicitly modelling sequence order, flattening the input still loses useful temporal relationships. Even so, the MLP remains reasonably competitive on this dataset, which implies that the weather series contains strong short-term continuity and smoother behaviour compared to stock returns.

The Transformer ranked last on the Weather dataset, with RMSE = 0.0846 and MAE = 0.0688. Although its results are still acceptable, it is clearly less effective than the recurrent models in this case. This may suggest that the added complexity of attention is not especially beneficial for this relatively structured forecasting task, where simpler sequential models already capture the main dependencies very well.

Overall, the Weather results show that the RNN is the most suitable model in the current experimental setting, followed closely by the LSTM. This dataset appears to reward stable temporal memory more than model complexity.

In [4]:
best_rmse = results_df.loc[results_df.groupby("dataset")["rmse"].idxmin()][
    ["dataset", "model", "rmse", "mae"]
].reset_index(drop=True)

best_mae = results_df.loc[results_df.groupby("dataset")["mae"].idxmin()][
    ["dataset", "model", "rmse", "mae"]
].reset_index(drop=True)

print("Best model by RMSE")
display(best_rmse)

print("Best model by MAE")
display(best_mae)

Best model by RMSE


,dataset,model,rmse,mae
0,apple,transformer,0.569930,0.386512
1,weather,rnn,0.036063,0.026517


Best model by MAE


,dataset,model,rmse,mae
0,apple,transformer,0.569930,0.386512
1,weather,rnn,0.036063,0.026517


### Best Model by RMSE

Using RMSE as the selection criterion, the best model for Apple is the Transformer, while the best model for Weather is the RNN. RMSE places greater emphasis on larger prediction errors because the errors are squared before averaging. Therefore, this metric is especially useful when we want to know which model handles larger deviations more effectively.

For Apple, the Transformer’s lowest RMSE suggests that it was the most effective at reducing larger forecasting errors in the log-return series. This is a strong result because financial returns are noisy and contain occasional sharp movements. The Transformer’s attention mechanism may have helped it capture more informative relationships within the input window.

For Weather, the RNN achieved the best RMSE, indicating that it handled the overall prediction task more reliably than the other models. Since weather signals often evolve smoothly over time, the recurrent hidden-state mechanism appears sufficient to capture the main dependencies without requiring the added complexity of LSTM gating or Transformer attention.

This result shows that the best model depends on the dataset: Transformer is strongest for Apple, while RNN is strongest for Weather when focusing on squared error performance.

### Best Model by MAE

Using MAE as the selection criterion leads to the same conclusion as RMSE: the Transformer is best for Apple, and the RNN is best for Weather. This consistency is important because it strengthens confidence in the ranking. Since MAE measures the average absolute error directly, it is easier to interpret and less sensitive to very large outliers than RMSE.

For Apple, the Transformer’s lowest MAE indicates that it produces the smallest average deviation from the true values across the test set. This means its advantage is not only due to handling a few large errors better, but also due to consistently making more accurate predictions overall. The LSTM is very close behind, which suggests both models are strong candidates for this task.

For Weather, the RNN again achieves the best MAE, confirming that it is the most accurate model on average. The small gap between the RNN and LSTM shows that both recurrent models are highly effective, but the RNN remains the strongest overall in this setup.

Because the best model under MAE is the same as under RMSE for both datasets, the model selection is stable and well-supported by the evaluation metrics. This gives a clearer basis for the final discussion and comparison section.

### Final Comparative Interpretation

The final results show that no single architecture is best for all time series tasks. For the Apple dataset, the Transformer performs best, followed closely by the LSTM, showing that more advanced sequence modelling is beneficial when forecasting noisy financial return data. For the Weather dataset, the RNN performs best, suggesting that simpler recurrent memory is sufficient for capturing the smoother and more regular temporal structure of meteorological data.

Across both datasets, the MLP is never the top model, which highlights the limitation of ignoring sequential order in time series forecasting. Overall, the results confirm that preserving temporal structure is important, but the most suitable sequence model depends on the characteristics of the dataset itself.